# Camera-Radar BEV Fusion — Treinamento

Este notebook executa o treinamento do BEVFusionDetector no Google Colab com GPU T4.

### Passos:
1. Montar Google Drive
2. Instalar dependencias
3. Baixar nuScenes mini
4. Treinar modelo
5. Avaliar resultados

In [ ]:
# ─── Célula 1: Verificar GPU ───
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ─── Célula 2: Montar Google Drive ───
from google.colab import drive
drive.mount('/content/drive')

# Ajustar este caminho conforme a localizacao do seu repositorio no Drive
PROJECT_PATH = '/content/drive/MyDrive/PIBIC/fusion'
import sys
sys.path.insert(0, PROJECT_PATH)
print(f'Projeto: {PROJECT_PATH}')

In [ ]:
# ─── Célula 3: Instalar dependencias ───
!pip install -q nuscenes-devkit einops pyyaml tqdm matplotlib tensorboard


In [ ]:

# ─── Célula 4: Baixar dataset ───
!wget https://www.nuscenes.org/data/v1.0-mini.tgz 

In [ ]:
# ─── Célula 4: Baixar nuScenes mini (só na primeira vez) ───
import os

NUSCENES_PATH = '/content/data/sets/nuscenes'

if not os.path.exists(f'{NUSCENES_PATH}/v1.0-mini'):
    print('Baixando nuScenes mini...')
    !mkdir -p /content/data/sets
    !wget -q https://www.nuscenes.org/data/v1.0-mini.tgz -O /content/v1.0-mini.tgz
    !tar -xzf /content/v1.0-mini.tgz -C /content/data/sets/
    !rm /content/v1.0-mini.tgz
    print('Download concluido!')
else:
    print('nuScenes mini ja existe.')

In [ ]:
# ─── Célula 5: Configurar paths ───
CONFIG_PATH = os.path.join(PROJECT_PATH, 'config', 'default.yaml')
print(f'Config: {CONFIG_PATH}')
print(f'nuScenes: {NUSCENES_PATH}')

In [ ]:
# ─── Célula 6: Treinar modelo ───
from src.engine.train import train

train(
    config_path=CONFIG_PATH,
    dataroot=NUSCENES_PATH,
)

In [ ]:
# ─── Célula 7: Avaliar modelo ───
from src.engine.evaluate import evaluate, print_metrics
from src.engine.train import load_config, build_model
from src.dataset.nuscenes_dataset import NuScenesFusionDataset, collate_fn
from torch.utils.data import DataLoader

cfg = load_config(CONFIG_PATH)
cfg['data']['dataroot'] = NUSCENES_PATH

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg, device)

# Carregar melhor modelo
CKPT_PATH = os.path.join(PROJECT_PATH, 'checkpoints', 'best_model.pth')
if os.path.exists(CKPT_PATH):
    from src.engine.train import load_checkpoint
    epoch, metrics = load_checkpoint(model, None, CKPT_PATH, device)
    print(f'Modelo carregado: epoch {epoch}')

    # Avaliar
    val_dataset = NuScenesFusionDataset(
        dataroot=NUSCENES_PATH,
        version='v1.0-mini',
        split='val',
        image_size=tuple(cfg['data']['image_size']),
        bev_size=tuple(cfg['data']['bev_size']),
        bev_range=tuple(cfg['data']['bev_range']),
        radar_max_points=cfg['data']['radar_max_points'],
    )
    val_loader = DataLoader(val_dataset, batch_size=4, collate_fn=collate_fn)
    results = evaluate(model, val_loader, device, cfg)
    print_metrics(results)
else:
    print('Checkpoint nao encontrado. Execute o treino primeiro.')

In [ ]:
# ─── Célula 8: Visualizar predicoes ───
import matplotlib.pyplot as plt
import numpy as np

model.eval()
sample = val_dataset[0]
image = sample['image'].unsqueeze(0).to(device)
radar_points = sample['radar_points'].unsqueeze(0).to(device)
radar_mask = sample['radar_mask'].unsqueeze(0).to(device)

# Radar → BEV
from src.dataset.radar_transforms import radar_to_bev_grid
radar_bev = radar_to_bev_grid(
    radar_points[0], radar_mask[0],
    bev_size=tuple(cfg['data']['bev_size']),
    bev_range=tuple(cfg['data']['bev_range']),
).unsqueeze(0).to(device)

# Predicao
with torch.no_grad():
    seg_logits = model(image, radar_bev)
    seg_prob = torch.sigmoid(seg_logits).cpu().squeeze().numpy()

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Imagem original
img_show = sample['image'].permute(1, 2, 0).numpy()
img_show = (img_show * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]))
img_show = np.clip(img_show, 0, 1)
axes[0].imshow(img_show)
axes[0].set_title('Imagem de Entrada')
axes[0].axis('off')

# Ground Truth
gt = sample['bev_segmentation'].squeeze().numpy()
axes[1].imshow(gt, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Ground Truth BEV')
axes[1].axis('off')

# Predicao
axes[2].imshow(seg_prob, cmap='gray', vmin=0, vmax=1)
axes[2].set_title('Predicao BEV')
axes[2].axis('off')

plt.tight_layout()
plt.show()